# EDA And Baseline Modeling

This notebook reviews the final analytics table and trains first baseline models for football player market value prediction.

Input table: `data/processed/player_season_analytics.csv`

Target: `log_market_value_eur`

In [ ]:
from pathlib import Path
import math

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 120)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")

In [ ]:
REPO_ROOT = Path.cwd().parent if Path.cwd().name == "analysis" else Path.cwd()
DATA_PATH = REPO_ROOT / "data" / "processed" / "player_season_analytics.csv"

TARGET = "log_market_value_eur"
RAW_TARGET = "market_value_eur"
TRAIN_SEASONS = ["21/22", "22/23"]
TEST_SEASONS = ["23/24"]

NUMERIC_FEATURES = [
    "Age",
    "age_squared",
    "Min",
    "Starts",
    "90s",
    "goals_per90",
    "assists_per90",
    "shots_per90",
    "shots_on_target_per90",
    "crosses_per90",
    "interceptions_per90",
    "tackles_won_per90",
    "fouls_per90",
    "fouled_per90",
    "contract_years_remaining",
    "contract_missing",
    "days_since_last_transfer",
    "transfer_count_before_valuation",
    "loan_count_before_valuation",
]

CATEGORICAL_FEATURES = ["position_group", "cleaned_comp", "season"]

df = pd.read_csv(DATA_PATH)
for column in sorted(set(NUMERIC_FEATURES + [TARGET, RAW_TARGET])):
    if column in df.columns:
        df[column] = pd.to_numeric(df[column], errors="coerce")

df.head()

## Dataset Overview

In [ ]:
overview = pd.DataFrame(
    {
        "metric": [
            "rows",
            "columns",
            "unique_players",
            "duplicate_row_ids",
            "missing_market_value",
            "missing_log_target",
            "missing_contract_rows",
            "rows_under_900_minutes",
        ],
        "value": [
            len(df),
            len(df.columns),
            df["trmrkt_player_id"].nunique(),
            df["row_id"].duplicated().sum(),
            df[RAW_TARGET].isna().sum(),
            df[TARGET].isna().sum(),
            (df["contract_missing"] == 1).sum(),
            (df["Min"] < 900).sum(),
        ],
    }
)
overview

In [ ]:
missingness = (
    df.isna()
    .sum()
    .rename("missing_count")
    .to_frame()
    .assign(missing_pct=lambda table: table["missing_count"] / len(df))
    .sort_values("missing_count", ascending=False)
)
missingness.head(20)

In [ ]:
display(df["season"].value_counts().rename_axis("season").reset_index(name="rows"))
display(df["cleaned_comp"].value_counts().rename_axis("league").reset_index(name="rows"))
display(df["position_group"].value_counts().rename_axis("position_group").reset_index(name="rows"))
display(df["minutes_bucket"].value_counts().rename_axis("minutes_bucket").reset_index(name="rows"))

## Target Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(df[RAW_TARGET].dropna(), bins=60, ax=axes[0])
axes[0].set_title("Market Value Distribution")
axes[0].set_xlabel("Market value EUR")

sns.histplot(df[TARGET].dropna(), bins=60, ax=axes[1])
axes[1].set_title("Log Market Value Distribution")
axes[1].set_xlabel("log1p(market value EUR)")

plt.tight_layout()

## Market Value By Context

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

league_order = df.groupby("cleaned_comp")[RAW_TARGET].median().sort_values(ascending=False).index
sns.boxplot(data=df, x="cleaned_comp", y=TARGET, order=league_order, ax=axes[0])
axes[0].set_title("Log Market Value By League")
axes[0].set_xlabel("League")
axes[0].set_ylabel("log1p(market value EUR)")
axes[0].tick_params(axis="x", rotation=25)

position_order = [position for position in ["DF", "MF", "FW", "GK"] if position in set(df["position_group"])]
sns.boxplot(data=df, x="position_group", y=TARGET, order=position_order, ax=axes[1])
axes[1].set_title("Log Market Value By Position")
axes[1].set_xlabel("Position group")
axes[1].set_ylabel("log1p(market value EUR)")

plt.tight_layout()

In [ ]:
market_value_by_group = []
for column in ["season", "cleaned_comp", "position_group", "minutes_bucket"]:
    table = (
        df.groupby(column, dropna=False)[RAW_TARGET]
        .agg(["count", "median", "mean"])
        .reset_index()
        .assign(group_column=column)
    )
    market_value_by_group.append(table)

pd.concat(market_value_by_group, ignore_index=True)

## Contract And Transfer-History Signals

In [ ]:
sample_df = df.sample(min(len(df), 2500), random_state=42)

plt.figure(figsize=(8, 5))
sns.scatterplot(
    data=sample_df,
    x="contract_years_remaining",
    y=TARGET,
    hue="position_group",
    alpha=0.55,
)
plt.title("Contract Years Remaining Vs Log Market Value")
plt.xlabel("Contract years remaining")
plt.ylabel("log1p(market value EUR)")
plt.tight_layout()

## Correlations

In [ ]:
correlation_columns = [
    RAW_TARGET,
    TARGET,
    "Age",
    "Min",
    "90s",
    "goals_per90",
    "assists_per90",
    "shots_per90",
    "shots_on_target_per90",
    "crosses_per90",
    "interceptions_per90",
    "tackles_won_per90",
    "contract_years_remaining",
    "days_since_last_transfer",
    "transfer_count_before_valuation",
]

corr = df[[column for column in correlation_columns if column in df.columns]].corr(numeric_only=True)
display(corr[TARGET].sort_values(ascending=False).to_frame("corr_with_log_market_value"))

plt.figure(figsize=(11, 8))
sns.heatmap(corr, cmap="vlag", center=0, linewidths=0.2)
plt.title("Numeric Feature Correlations")
plt.tight_layout()

## Modeling Dataset

The first modeling pass filters to players with at least 900 minutes. The split is by season to test whether the model generalizes to the next season.

In [ ]:
MIN_MINUTES = 900

model_df = df.copy()
model_df = model_df[model_df[TARGET].notna() & model_df[RAW_TARGET].notna()]
model_df = model_df[model_df["season"].isin(TRAIN_SEASONS + TEST_SEASONS)]
model_df = model_df[model_df["Min"].fillna(0) >= MIN_MINUTES]

for column in CATEGORICAL_FEATURES:
    model_df[column] = model_df[column].fillna("Unknown").astype(str)

split_summary = model_df.groupby("season").size().rename("rows").reset_index()
display(split_summary)
print(f"Modeling rows: {len(model_df):,}")

## Baseline Models

In [ ]:
def make_preprocessor():
    try:
        encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        encoder = OneHotEncoder(handle_unknown="ignore", sparse=False)

    numeric_pipeline = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]
    )
    categorical_pipeline = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", encoder),
        ]
    )

    return ColumnTransformer(
        transformers=[
            ("num", numeric_pipeline, NUMERIC_FEATURES),
            ("cat", categorical_pipeline, CATEGORICAL_FEATURES),
        ],
        remainder="drop",
    )


def evaluate_model(name, model, X, y, split):
    predictions = model.predict(X)
    return {
        "model": name,
        "split": split,
        "rows": len(y),
        "rmse_log": math.sqrt(mean_squared_error(y, predictions)),
        "mae_log": mean_absolute_error(y, predictions),
        "r2": r2_score(y, predictions),
    }

In [ ]:
train_df = model_df[model_df["season"].isin(TRAIN_SEASONS)].copy()
test_df = model_df[model_df["season"].isin(TEST_SEASONS)].copy()

X_train = train_df[NUMERIC_FEATURES + CATEGORICAL_FEATURES]
y_train = train_df[TARGET]
X_test = test_df[NUMERIC_FEATURES + CATEGORICAL_FEATURES]
y_test = test_df[TARGET]

models = [
    ("mean_baseline", Pipeline(steps=[("model", DummyRegressor(strategy="mean"))])),
    ("linear_regression", Pipeline(steps=[("preprocess", make_preprocessor()), ("model", LinearRegression())])),
    ("ridge_alpha_1", Pipeline(steps=[("preprocess", make_preprocessor()), ("model", Ridge(alpha=1.0))])),
    ("ridge_alpha_10", Pipeline(steps=[("preprocess", make_preprocessor()), ("model", Ridge(alpha=10.0))])),
]

metric_rows = []
fitted_models = {}

for name, model in models:
    model.fit(X_train, y_train)
    fitted_models[name] = model
    metric_rows.append(evaluate_model(name, model, X_train, y_train, "train"))
    metric_rows.append(evaluate_model(name, model, X_test, y_test, "test"))

metrics = pd.DataFrame(metric_rows)
metrics

## Ridge Coefficients

Ridge coefficients are not causal effects, but they are useful as an early interpretability check because numeric features are standardized and categorical features are one-hot encoded.

In [ ]:
ridge_model = fitted_models["ridge_alpha_1"]
preprocessor = ridge_model.named_steps["preprocess"]
estimator = ridge_model.named_steps["model"]

coefficients = (
    pd.DataFrame(
        {
            "feature": preprocessor.get_feature_names_out(),
            "coefficient": estimator.coef_,
            "abs_coefficient": np.abs(estimator.coef_),
        }
    )
    .sort_values("abs_coefficient", ascending=False)
    .reset_index(drop=True)
)

coefficients.head(20)

## First Takeaways

- The mean baseline is the minimum benchmark.
- Linear and ridge regression should beat it meaningfully if the dataset has useful signal.
- Early coefficient checks should be treated as descriptive, not causal.
- The next modeling step should add stronger nonlinear models and compare performance by position and league.